# UdeA Insurance — Reto Tarifación Seguro de Salud
## Avance 1: Limpieza de Datos y Análisis Exploratorio

### Bases de datos
- **BD_Exposicion:** Periodos de cobertura de cada asegurado 
- **BD_SocioDemografica:** Perfil demográfico y condiciones preexistentes 
- **BD_Siniestros:** Reclamaciones médicas históricas 

In [1]:
# Librerías estándar de análisis de datos
import pandas as pd
import numpy as np
from pathlib import Path

# Visualización
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


In [2]:
#Obtener la raíz del proyecto (subiendo un nivel desde la carpeta src/)
proyecto_root = Path.cwd().parent

# Ruta a la carpeta data
data_dir = proyecto_root / "data"

## 2. Limpieza: BD_Exposicion

### ¿Qué hace esta base de datos?
Registra **cuánto tiempo estuvo asegurada cada persona**. Esto es fundamental
porque en actuaría no importa solo cuánto costó una persona, sino cuánto costó
*por unidad de tiempo expuesta*. Ese indicador se llama **tasa de siniestralidad**.

### Pasos que vamos a ejecutar:
1. Diagnóstico inicial (tipos, nulos, duplicados)
2. Conversión de fechas (de texto a datetime)
3. Calcular la **fecha de salida real** (cancelación o fin de contrato)
4. Calcular la **exposición** en días y meses
5. Detectar y reportar inconsistencias

# Paso 1: Diagnóstico inicial (tipos, nulos, duplicados)


In [3]:
# Leer uno de los archivos 
df_exposicion = pd.read_csv(data_dir / "BD_Exposicion.txt", sep="\t")  # o sep=',' o sep=';'

# Diagnóstico inicial ──────────────────────────────────────────────────
print('Primeras cinco filas ')
display(df_exposicion.head(5))

print('\n Tipos de datos y nulos por columna')
print(df_exposicion.info())

print('\n Valores nulos por columna y porcentaje')
nulos = df_exposicion.isnull().sum()
pct   = (nulos / len(df_exposicion) * 100).round(2)
print(pd.DataFrame({'nulos': nulos, 'pct_%': pct}))

print(f'\nFilas duplicadas: {df_exposicion.duplicated().sum():,}')
print(f'Asegurados únicos (Asegurado_Id): {df_exposicion["Asegurado_Id"].nunique():,}')
print(f'Pólizas únicas  (Poliza_Asegurado_Id): {df_exposicion["Poliza_Asegurado_Id"].nunique():,}')


Primeras cinco filas 


,Asegurado_Id,Poliza_Asegurado_Id,FECHA_INICIO,FECHA_CANCELACION,FECHA_FIN
0,61630146,271191574,29/01/2024,29/06/2024,29/06/2024
1,7966225,283714899,13/11/2024,NaN,31/12/2024
2,3124492,50801309,1/01/2024,17/01/2024,17/01/2024
3,11863430,236507980,1/01/2024,NaN,31/12/2024
4,4345756,230314288,1/01/2024,NaN,31/12/2024



 Tipos de datos y nulos por columna
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320002 entries, 0 to 320001
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   Asegurado_Id         320002 non-null  int64 
 1   Poliza_Asegurado_Id  320002 non-null  int64 
 2   FECHA_INICIO         320002 non-null  object
 3   FECHA_CANCELACION    109031 non-null  object
 4   FECHA_FIN            320002 non-null  object
dtypes: int64(2), object(3)
memory usage: 12.2+ MB
None

 Valores nulos por columna y porcentaje
                      nulos  pct_%
Asegurado_Id              0   0.00
Poliza_Asegurado_Id       0   0.00
FECHA_INICIO              0   0.00
FECHA_CANCELACION    210971  65.93
FECHA_FIN                 0   0.00

Filas duplicadas: 0
Asegurados únicos (Asegurado_Id): 296,020
Pólizas únicas  (Poliza_Asegurado_Id): 320,002


# Paso 2: Conversión de fechas (de texto a datetime)

In [4]:
# Convertir columnas de fecha antes de operar con ellas
for columna in ['FECHA_INICIO', 'FECHA_FIN', 'FECHA_CANCELACION']:
    df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')

print('Columnas de fecha convertidas a datetime correctamente.')

Columnas de fecha convertidas a datetime correctamente.


/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_1487/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')
/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_1487/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')
/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_1487/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')


# Paso 3: Calcular la **fecha de salida real** (cancelación o fin de contrato)


In [5]:
# Fecha de salida real ─────────────────────────────────────────────────

df_exposicion['FECHA_SALIDA'] = df_exposicion['FECHA_CANCELACION'].fillna(df_exposicion['FECHA_FIN'])

# Validación extra: si FECHA_CANCELACION ES MAYOR A  FECHA_FIN, hay un error en los datos
# (la cancelación ocurrió DESPUÉS del fin del contrato, lo cual no tiene sentido)
mask_error = (
    df_exposicion['FECHA_CANCELACION'].notna() &
    (df_exposicion['FECHA_CANCELACION'] > df_exposicion['FECHA_FIN'])
)
print(f'Registros con FECHA_CANCELACION > FECHA_FIN (inconsistentes): {mask_error.sum():,}')

# Para esos casos, corregimos usando FECHA_FIN como salida real
df_exposicion.loc[mask_error, 'FECHA_SALIDA'] = df_exposicion.loc[mask_error, 'FECHA_FIN']

print('\n Ejemplo de la nueva columna FECHA_SALIDA:')
display(df_exposicion[['Asegurado_Id', 'FECHA_INICIO', 'FECHA_CANCELACION', 'FECHA_FIN', 'FECHA_SALIDA']].head(8))

Registros con FECHA_CANCELACION > FECHA_FIN (inconsistentes): 18

 Ejemplo de la nueva columna FECHA_SALIDA:


,Asegurado_Id,FECHA_INICIO,FECHA_CANCELACION,FECHA_FIN,FECHA_SALIDA
0,61630146,2024-01-29,2024-06-29,2024-06-29,2024-06-29
1,7966225,2024-11-13,NaT,2024-12-31,2024-12-31
2,3124492,2024-01-01,2024-01-17,2024-01-17,2024-01-17
3,11863430,2024-01-01,NaT,2024-12-31,2024-12-31
4,4345756,2024-01-01,NaT,2024-12-31,2024-12-31
5,8134114,2024-01-01,NaT,2024-12-31,2024-12-31
6,34285636,2024-01-01,2024-02-18,2024-02-18,2024-02-18
7,54470034,2024-01-01,2024-12-31,2024-12-31,2024-12-31


# Paso 4: Calcular la **exposición** en días y meses

In [6]:
#Calcular exposición ──────────────────────────────────────────────────

df_exposicion['dias_expuesto'] = (df_exposicion['FECHA_SALIDA'] - df_exposicion['FECHA_INICIO']).dt.days

# Convertir a meses (aproximación: 1 mes = 30.4375 días)
df_exposicion['meses_expuesto'] = df_exposicion['dias_expuesto'] / 30.4375

# Detectar exposiciones inválidas
print(' Diagnóstico de exposición ')
print(f'Exposición negativa (error de fechas):  {(df_exposicion["dias_expuesto"] < 0).sum():,}')
print(f'Exposición = 0 días:                    {(df_exposicion["dias_expuesto"] == 0).sum():,}')
print(f'Exposición > 365 días (más de 1 año):   {(df_exposicion["dias_expuesto"] > 365).sum():,}')
print()
print(df_exposicion['dias_expuesto'].describe())

 Diagnóstico de exposición 
Exposición negativa (error de fechas):  0
Exposición = 0 días:                    6,578
Exposición > 365 días (más de 1 año):   0

count    320002.000000
mean        294.559943
std         116.671127
min           0.000000
25%         244.000000
50%         365.000000
75%         365.000000
max         365.000000
Name: dias_expuesto, dtype: float64


# Paso 5: Detectar y reportar inconsistencias

In [7]:
# Consolidación final de BD_Exposicion (modelado) ────────────────────────────
# 1) Filtrar exposición válida
n_invalidos = (df_exposicion['dias_expuesto'] <= 0).sum()
pct_invalidos = n_invalidos / len(df_exposicion) * 100

print(f'Exposición <= 0: {n_invalidos:,} ({pct_invalidos:.2f}%)')

df_exposicion['exposicion_valida'] = df_exposicion['dias_expuesto'] > 0
df_exposicion_limpio = df_exposicion[df_exposicion['exposicion_valida']].copy()

# 2) Diagnóstico de duplicados
dup_exactos = df_exposicion_limpio.duplicated().sum()
dup_poliza_fechas = df_exposicion_limpio.duplicated(
    subset=['Poliza_Asegurado_Id', 'FECHA_INICIO', 'FECHA_SALIDA']
).sum()
dup_poliza = df_exposicion_limpio.duplicated(subset=['Poliza_Asegurado_Id']).sum()

print('\n=== Resumen final de BD_Exposicion ===')
print(f'Duplicados exactos: {dup_exactos:,}')
print(f'Duplicados por póliza+fechas: {dup_poliza_fechas:,}')
print(f'Pólizas repetidas (cualquier periodo): {dup_poliza:,}')

# 3) Base final de modelado (remueve solo duplicado exacto)
df_exposicion_modelo = df_exposicion_limpio.drop_duplicates().copy()

print('\n=== información final para modelado ===')
print(f'Filas finales: {len(df_exposicion_modelo):,}')
print(f'Asegurados únicos: {df_exposicion_modelo["Asegurado_Id"].nunique():,}')
print(f'Pólizas únicas: {df_exposicion_modelo["Poliza_Asegurado_Id"].nunique():,}')

Exposición <= 0: 6,578 (2.06%)

=== Resumen final de BD_Exposicion ===
Duplicados exactos: 0
Duplicados por póliza+fechas: 0
Pólizas repetidas (cualquier periodo): 0

=== información final para modelado ===
Filas finales: 313,424
Asegurados únicos: 293,288
Pólizas únicas: 313,424


## 3. Limpieza: BD_SocioDemografica

### ¿Qué hace esta base de datos?
Contiene el **perfil de riesgo** de cada asegurado (UNA fila por Afiliado_Id). Aquí están las variables que más influyen en el costo de un seguro de salud: edad, sexo, ciudad y enfermedades preexistentes.

### Estructura de 5 pasos:
1. **Paso 1**: Diagnóstico inicial (nulos, tipos, duplicados)
2. **Paso 2**: Convertir FechaNacimiento a datetime y corregir años futuros
3. **Paso 3**: Calcular edad y crear grupos de edad
4. **Paso 4**: Crear variables derivadas (num_condiciones, Sexo_Cd_limpio, CIUDAD_NORM)
5. **Paso 5**: Validación y cierre final (manejo de duplicados, df_sociodemograficos_limpio listo para EDA)

# Paso 1: Diagnóstico Inicial
Nulos, tipos, duplicados por Afiliado_Id

In [8]:
# Leer la base de sociodemográficos (ajusta el separador si hace falta)
df_sociodemograficos = pd.read_csv(data_dir / "BD_SocioDemografica.txt", sep="\t", encoding="latin1")

# ── 3.1 Diagnóstico inicial ──────────────────────────────────────────────────
print('Primeras filas de la bd_sociodemograficos:')
display(df_sociodemograficos.head())

print('\n Informacion general ')
df_sociodemograficos.info()

print('\n  Valores únicos en Sexo_Cd ')
print(df_sociodemograficos['Sexo_Cd'].value_counts())

print('\nDuplicados por Afiliado_Id ')
print(f'Duplicados encontrados: {df_sociodemograficos.duplicated(subset=["Afiliado_Id"]).sum()}')



Primeras filas de la bd_sociodemograficos:


,Afiliado_Id,Sexo_Cd,FechaNacimiento,CIUDAD,CANCER,DIABETES,ENF_CARDIACA,HIPERTENSION,ENF_PULMONAR
0,921437,F,30/04/68,Medellin,0,0,0,0,0
1,60504878,M,18/02/12,Medellin,0,0,0,0,0
2,55074222,F,23/10/14,Medellin,0,0,0,0,0
3,23690690,F,27/06/89,Cartagena,0,0,0,0,0
4,45506882,M,30/06/09,Cali,0,0,0,0,0



 Informacion general 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261267 entries, 0 to 261266
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Afiliado_Id      261267 non-null  int64 
 1   Sexo_Cd          261267 non-null  object
 2   FechaNacimiento  261267 non-null  object
 3   CIUDAD           261267 non-null  object
 4   CANCER           261267 non-null  int64 
 5   DIABETES         261267 non-null  int64 
 6   ENF_CARDIACA     261267 non-null  int64 
 7   HIPERTENSION     261267 non-null  int64 
 8   ENF_PULMONAR     261267 non-null  int64 
dtypes: int64(6), object(3)
memory usage: 17.9+ MB

  Valores únicos en Sexo_Cd 
Sexo_Cd
F     140329
M     120902
-1        36
Name: count, dtype: int64

Duplicados por Afiliado_Id 
Duplicados encontrados: 2


In [9]:
# Identificar duplicados exactos por Afiliado_Id y eliminar
n_antes = len(df_sociodemograficos)
print(display(df_sociodemograficos[df_sociodemograficos.duplicated(subset=['Afiliado_Id'], keep=False)]))

,Afiliado_Id,Sexo_Cd,FechaNacimiento,CIUDAD,CANCER,DIABETES,ENF_CARDIACA,HIPERTENSION,ENF_PULMONAR
32990,62942960,F,24/03/25,Cartagena,0,0,0,0,0
80214,58987622,F,10/10/14,Medellin,0,0,0,0,0
165000,62942960,F,24/03/25,Bogota,0,0,0,0,0
240541,58987622,F,10/10/14,Medellin,0,0,0,0,0


None


In [10]:
# Eliminación de Duplicados Exactos
df_sociodemograficos = df_sociodemograficos.drop_duplicates(subset=['Afiliado_Id'], keep='first')

n_despues = len(df_sociodemograficos)
n_eliminados = n_antes - n_despues

print('ELIMINACIÓN DE DUPLICADOS EXACTOS')
print(f'Filas antes:    {n_antes:,}')
print(f'Filas después:  {n_despues:,}')
print(f'Eliminadas:     {n_eliminados}')


ELIMINACIÓN DE DUPLICADOS EXACTOS
Filas antes:    261,267
Filas después:  261,265
Eliminadas:     2


# Paso 2: Convertir Fechas
Convertir FechaNacimiento a datetime y corregir años futuros (parseo de 2 dígitos)

In [11]:
# Convertir FechaNacimiento a datetime 
df_sociodemograficos['FechaNacimiento_dt'] = pd.to_datetime(
    df_sociodemograficos['FechaNacimiento'],
    dayfirst=True,
    errors='coerce'
)

print('Fechas convertidas correctamente.')
print(f"Fechas no parseadas (NaT): {df_sociodemograficos['FechaNacimiento_dt'].isna().sum():,}")

/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_1487/651047863.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_sociodemograficos['FechaNacimiento_dt'] = pd.to_datetime(


Fechas convertidas correctamente.
Fechas no parseadas (NaT): 0


# Paso 3: Calcular Edad y Grupos de Edad

**edad**: Años cumplidos al 2026-04-01

**grupo_edad**: Binning en 6 categorías (0-17, 18-30, 31-45, 46-60, 61-75, 76+)

**num_condiciones**: Suma de 5 enfermedades preexistentes (CANCER, DIABETES, ENF_CARDIACA, HIPERTENSION, ENF_PULMONAR)  
- 0 = ninguna enfermedad
- 1-5 = número de enfermedades presentes

In [12]:
#Calcular edad desde FechaNacimiento_dt
FECHA_CORTE = pd.Timestamp('2026-04-01')

# Ajuste para años futuros por parseo de fechas con 2 dígitos
mask_futuro = df_sociodemograficos['FechaNacimiento_dt'] > FECHA_CORTE
df_sociodemograficos.loc[mask_futuro, 'FechaNacimiento_dt'] = (
    df_sociodemograficos.loc[mask_futuro, 'FechaNacimiento_dt'] - pd.DateOffset(years=100)
)

# Edad entera: con redondeo hacia abajo en años cumplidos
df_sociodemograficos['edad'] = np.floor(
    (FECHA_CORTE - df_sociodemograficos['FechaNacimiento_dt']).dt.days / 365.25
).astype('Int64')

print('Edad calculada correctamente.')
print(f"Edades nulas: {df_sociodemograficos['edad'].isna().sum():,}")
print(f"Edades < 0: {(df_sociodemograficos['edad'] < 0).sum():,}")
print(f"Edades > 100: {(df_sociodemograficos['edad'] > 100).sum():,}")

Edad calculada correctamente.
Edades nulas: 0
Edades < 0: 0
Edades > 100: 0


In [13]:
#Grupos de edad (bins) para análisis posterior
bins = [0, 17, 30, 45, 60, 75, 120]
labels = ['0-17', '18-30', '31-45', '46-60', '61-75', '76+']

df_sociodemograficos["grupo_edad"] = pd.cut(
    df_sociodemograficos["edad"],
    bins=[0, 17, 30, 45, 60, 75, 120],
    labels=["0-17", "18-30", "31-45", "46-60", "61-75", "76+"],
    right=True,
    include_lowest=True
)

# Número de condiciones preexistentes
cols_binarias = ['CANCER', 'DIABETES', 'ENF_CARDIACA', 'HIPERTENSION', 'ENF_PULMONAR']
df_sociodemograficos['num_condiciones'] = df_sociodemograficos[cols_binarias].sum(axis=1)

print('Variables derivadas creadas')
print(df_sociodemograficos['grupo_edad'].value_counts(dropna=False).sort_index())
print('\nDistribución de num_condiciones:')
print(df_sociodemograficos['num_condiciones'].value_counts().sort_index())



Variables derivadas creadas
grupo_edad
0-17     57915
18-30    38036
31-45    86612
46-60    50767
61-75    23163
76+       4772
Name: count, dtype: int64

Distribución de num_condiciones:
num_condiciones
0    226859
1     27804
2      5391
3      1054
4       144
5        13
Name: count, dtype: int64


# Paso 4: Crear Variables Derivadas
Normalizar CIUDAD (mayúsculas, sin acentos), Codificar Sexo_Cd_limpio (F|M|DESCONOCIDO)

In [14]:
# Validación y Cierre Final

import unicodedata

print('\nNORMALIZACION VARIABLE CIUDAD')
if 'CIUDAD_NORM' not in df_sociodemograficos.columns:
    def normalizar_texto(texto):
        if pd.isna(texto):
            return texto
        texto = str(texto).strip().upper()
        texto = unicodedata.normalize('NFD', texto)
        texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
        return texto

    df_sociodemograficos['CIUDAD_NORM'] = df_sociodemograficos['CIUDAD'].apply(normalizar_texto)

print('Ejemplo de normalizacion de CIUDAD:')
print(df_sociodemograficos[['CIUDAD', 'CIUDAD_NORM']].drop_duplicates().head(10))

print('\nCODIFICAR VARIABLES CATEGORICAS (SEXO):')
df_sociodemograficos['Sexo_Cd_limpio'] = (
    df_sociodemograficos['Sexo_Cd'].astype(str).str.upper().where(
        df_sociodemograficos['Sexo_Cd'].astype(str).str.upper().isin(['F', 'M']),
        'NOBINARIO'
    )
)

print('Valores Sexo_Cd_limpio:')
print(df_sociodemograficos['Sexo_Cd_limpio'].value_counts().to_string())

# Consolidacion final con columnas acordadas
columnas_finales = [
    'Afiliado_Id',
    'FechaNacimiento_dt',
    'edad',
    'grupo_edad',
    'CIUDAD_NORM',
    'Sexo_Cd_limpio',
    'CANCER',
    'DIABETES',
    'ENF_CARDIACA',
    'HIPERTENSION',
    'ENF_PULMONAR',
    'num_condiciones'
]

df_sociodemograficos_limpio = df_sociodemograficos[columnas_finales].copy()

print('\nBase final creada: df_sociodemograficos_limpio')


NORMALIZACION VARIABLE CIUDAD
Ejemplo de normalizacion de CIUDAD:
               CIUDAD      CIUDAD_NORM
0            Medellin         MEDELLIN
3           Cartagena        CARTAGENA
4                Cali             CALI
6              Bogota           BOGOTA
2525  Sin Informacion  SIN INFORMACION

CODIFICAR VARIABLES CATEGORICAS (SEXO):
Valores Sexo_Cd_limpio:
Sexo_Cd_limpio
F            140327
M            120902
NOBINARIO        36

Base final creada: df_sociodemograficos_limpio


In [15]:
df_sociodemograficos_limpio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 261265 entries, 0 to 261266
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Afiliado_Id         261265 non-null  int64         
 1   FechaNacimiento_dt  261265 non-null  datetime64[ns]
 2   edad                261265 non-null  Int64         
 3   grupo_edad          261265 non-null  category      
 4   CIUDAD_NORM         261265 non-null  object        
 5   Sexo_Cd_limpio      261265 non-null  object        
 6   CANCER              261265 non-null  int64         
 7   DIABETES            261265 non-null  int64         
 8   ENF_CARDIACA        261265 non-null  int64         
 9   HIPERTENSION        261265 non-null  int64         
 10  ENF_PULMONAR        261265 non-null  int64         
 11  num_condiciones     261265 non-null  int64         
dtypes: Int64(1), category(1), datetime64[ns](1), int64(7), object(2)
memory usage: 24.4+ MB


## 4. Limpieza: BD_Siniestros

### ¿Qué hace esta base de datos?
Contiene cada **reclamación médica individual**: quién reclamó, cuándo, qué diagnóstico
tenía y cuánto costó. Es la base del modelo de predicción de costos.

### Pasos a ejecutar:
1. Diagnóstico inicial
2. Convertir fecha de reclamación
3. Analizar y limpiar Valor_Pagado
4. Explorar diagnósticos CIE-10
5. Explorar tipos de reclamación
6. Agregar por asegurado (para el join posterior)

# Paso 1: Diagnóstico inicial 
Información general de la base de datos, tipo de datos, datos nulos, datos duplicados etc..

In [26]:
# Leer la base de siniestros (ajusta el separador si hace falta)
df_siniestros = pd.read_csv(data_dir / "BD_Siniestros.txt", sep="\t", encoding="latin1")

# Diagnóstico inicial ──────────────────────────────────────────────────
print('=== Primeras filas ===')
display(df_siniestros.head(5))

print('\n=== Info ===')
print(df_siniestros.info())

print('\n=== Resumen de nulos y faltantes ===')
columnas_texto = df_siniestros.select_dtypes(include=['object', 'string']).columns
columnas_num = df_siniestros.columns.difference(columnas_texto)

nulos_sin = df_siniestros.isna().sum()
pct_sin = (nulos_sin / len(df_siniestros) * 100).round(3)

faltantes_texto = df_siniestros[columnas_texto].apply(
    lambda s: s.astype('string').str.strip().eq('').sum()
)

resumen = pd.DataFrame({
    'dtype': df_siniestros.dtypes.astype(str),
    'nulos': nulos_sin,
    'pct_nulos': pct_sin,
    'faltantes_texto': 0
})
resumen.loc[columnas_texto, 'faltantes_texto'] = faltantes_texto
resumen['faltantes_totales'] = resumen['nulos'] + resumen['faltantes_texto']

print('--- Columnas numéricas / fecha ---')
display(resumen.loc[columnas_num].sort_values('nulos', ascending=False))

print('--- Columnas de texto ---')
display(resumen.loc[columnas_texto].sort_values('faltantes_totales', ascending=False))

=== Primeras filas ===


,Fecha_Reclamacion,Afiliado_Id,Reclamacion,Diagnostico_Codigo,Diagnostico_Desc,Eventos,Valor_Pagado
0,4/12/2024,31586355,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,101576.02545
1,15/01/2024,22753623,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,80630.89152
2,27/05/2024,29321930,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,90237.30633
3,6/11/2024,5258725,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,90237.30633
4,19/07/2024,22815238,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,8346.55713



=== Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1919949 entries, 0 to 1919948
Data columns (total 7 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Fecha_Reclamacion   object 
 1   Afiliado_Id         int64  
 2   Reclamacion         object 
 3   Diagnostico_Codigo  object 
 4   Diagnostico_Desc    object 
 5   Eventos             int64  
 6   Valor_Pagado        float64
dtypes: float64(1), int64(2), object(4)
memory usage: 102.5+ MB
None

=== Resumen de nulos y faltantes ===
--- Columnas numéricas / fecha ---


,dtype,nulos,pct_nulos,faltantes_texto,faltantes_totales
Afiliado_Id,int64,0,0.0,0,0
Eventos,int64,0,0.0,0,0
Valor_Pagado,float64,0,0.0,0,0


--- Columnas de texto ---


,dtype,nulos,pct_nulos,faltantes_texto,faltantes_totales
Fecha_Reclamacion,object,0,0.0,0,0
Reclamacion,object,0,0.0,0,0
Diagnostico_Codigo,object,0,0.0,0,0
Diagnostico_Desc,object,0,0.0,0,0


# Paso 2: Convertir Fecha_Reclamación 

In [17]:
df_siniestros['Fecha_Reclamacion'] = pd.to_datetime(
    df_siniestros['Fecha_Reclamacion'], dayfirst=True, errors='coerce'
)

print('Rango de fechas de reclamaciones:')
print(f'  Mínima: {df_siniestros["Fecha_Reclamacion"].min()}')
print(f'  Máxima: {df_siniestros["Fecha_Reclamacion"].max()}')
print(f'  NaT (no parseadas): {df_siniestros["Fecha_Reclamacion"].isna().sum():,}')

Rango de fechas de reclamaciones:
  Mínima: 2024-01-01 00:00:00
  Máxima: 2024-12-31 00:00:00
  NaT (no parseadas): 0


Codigo para validar si existen datos duplicados exactos

In [19]:
dup_exactos = df_siniestros.duplicated().sum()

llave_negocio = ['Afiliado_Id', 'Fecha_Reclamacion', 'Reclamacion',
'Diagnostico_Codigo', 'Valor_Pagado']
dup_negocio = df_siniestros.duplicated(subset=llave_negocio).sum()

print('Duplicados exactos:', dup_exactos)
print('Duplicados de negocio:', dup_negocio)

Duplicados exactos: 0
Duplicados de negocio: 0


Estandarizacion  de variables texto

In [20]:
for c in ['Reclamacion', 'Diagnostico_Codigo', 'Diagnostico_Desc']:
    df_siniestros[c] = (
df_siniestros[c].astype(str).str.strip().str.upper()
)

Revisar por variable texto (Objeto) cantidad de categorias unicas.

In [29]:
cols = ['Reclamacion', 'Diagnostico_Codigo', 'Diagnostico_Desc']

for c in cols:
    print('\n===', c, '===')
    print('Categorías únicas:', df_siniestros[c].nunique(dropna=False))
    print(df_siniestros[c].value_counts(dropna=False).head(20))


=== Reclamacion ===
Categorías únicas: 8
Reclamacion
CONSULTA MEDICA                1212864
EXAMENES MEDICOS                570237
CIRUGIA                          77008
LABORATORIO                      28543
HOSPITALIZACIONES SIMPLES        13339
HOSPITALIZACIONES COMPLEJAS       8557
CONSULTAS                         6714
TRATAMIENTO DE CANCER             2687
Name: count, dtype: int64

=== Diagnostico_Codigo ===
Categorías únicas: 4372
Diagnostico_Codigo
9       1585021
Z017      87697
Z108      38210
Z014      10799
Z000       9988
I10X       7846
R688       7777
Z321       5485
R529       5292
Z008       5101
L82X       3770
D239       3657
Z019       3460
B07X       3458
Z018       2916
L570       2770
Z011       1922
N390       1863
Z123       1862
D229       1839
Name: count, dtype: int64

=== Diagnostico_Desc ===
Categorías únicas: 4333
Diagnostico_Desc
DIAGNÓSTICO PENDIENTE                                                             1585025
EXAMEN DE LABORATORIO             

In [30]:
print("=== Validación de Valor_Pagado y Eventos ===")

print("\n--- Valor_Pagado ---")
print(df_siniestros["Valor_Pagado"].describe(percentiles=[.5, .75, .9, .95, .99]))
print("Valores negativos:", (df_siniestros["Valor_Pagado"] < 0).sum())
print("Valores iguales a 0:", (df_siniestros["Valor_Pagado"] == 0).sum())

print("\n--- Eventos ---")
print(df_siniestros["Eventos"].describe(percentiles=[.5, .75, .9, .95, .99]))
print("Valores menores o iguales a 0:", (df_siniestros["Eventos"] <= 0).sum())

=== Validación de Valor_Pagado y Eventos ===

--- Valor_Pagado ---
count    1.919949e+06
mean     6.593518e+05
std      5.644707e+06
min      1.574822e+00
50%      1.105525e+05
75%      2.278768e+05
90%      8.571026e+05
95%      1.730479e+06
99%      1.017384e+07
max      1.330849e+09
Name: Valor_Pagado, dtype: float64
Valores negativos: 0
Valores iguales a 0: 0

--- Eventos ---
count    1.919949e+06
mean     1.151845e+00
std      1.284459e+00
min      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      1.000000e+00
95%      2.000000e+00
99%      3.000000e+00
max      6.300000e+01
Name: Eventos, dtype: float64
Valores menores o iguales a 0: 0


### Paso 3: Dummizacion de tipo de reclamacion
Convertimos `Reclamacion` a columnas `rec_*` a nivel de fila de siniestro.

In [31]:
# Dummies por tipo de reclamacion (a nivel de fila de siniestro)
dummies_reclamacion = pd.get_dummies(df_siniestros['Reclamacion'], prefix='rec', dtype=int)

# Agregar dummies por afiliado (conteos)
df_reclamacion_agg = (
    pd.concat([df_siniestros[['Afiliado_Id']], dummies_reclamacion], axis=1)
    .groupby('Afiliado_Id', as_index=False)
    .sum()
)

print('Dummizacion lista: df_reclamacion_agg')
print(f'Filas: {len(df_reclamacion_agg):,}')
print(f'Tipos de reclamacion convertidos a columnas: {len([c for c in df_reclamacion_agg.columns if c.startswith("rec_")])}')
display(df_reclamacion_agg.head(10))

Dummizacion lista: df_reclamacion_agg
Filas: 240,108
Tipos de reclamacion convertidos a columnas: 8


,Afiliado_Id,rec_CIRUGIA,rec_CONSULTA MEDICA,rec_CONSULTAS,rec_EXAMENES MEDICOS,rec_HOSPITALIZACIONES COMPLEJAS,rec_HOSPITALIZACIONES SIMPLES,rec_LABORATORIO,rec_TRATAMIENTO DE CANCER
0,738294,0,4,0,4,0,0,0,0
1,738303,0,1,0,6,0,0,0,0
2,738305,0,6,0,1,0,0,0,0
3,738310,1,5,0,3,0,0,0,0
4,738318,0,0,0,3,0,0,0,0
5,738321,2,4,0,6,0,0,0,0
6,738348,0,5,0,4,0,0,0,0
7,738349,0,1,2,1,0,1,0,0
8,738358,0,11,0,1,0,0,0,0
9,738364,3,7,0,2,0,0,0,0


# Paso 4: Agrupar por Afiliado_Id
La base de siniestros está a nivel de reclamación; aquí la resumimos a nivel de asegurado para poder unirla luego con exposición y sociodemográficos.

In [33]:
# Paso 4.2: Agregado minimo por Afiliado_Id e integracion de dummies

df_siniestros_agg = df_siniestros.groupby('Afiliado_Id', as_index=False).agg(
    n_reclamaciones=('Reclamacion', 'size'),
    n_eventos=('Eventos', 'sum'),
    total_pagado=('Valor_Pagado', 'sum'),
    fecha_primera_reclamacion=('Fecha_Reclamacion', 'min'),
    fecha_ultima_reclamacion=('Fecha_Reclamacion', 'max')
)

# Unir dummies de reclamacion agregadas por afiliado
df_siniestros_agg = df_siniestros_agg.merge(df_reclamacion_agg, on='Afiliado_Id', how='left')

print('Base agregada: df_siniestros_agg')
print(f'Filas finales: {len(df_siniestros_agg):,}')
print(f'Asegurados unicos: {df_siniestros_agg["Afiliado_Id"].nunique():,}')
print(f'Columnas de tipo reclamacion agregadas: {len([c for c in df_siniestros_agg.columns if c.startswith("rec_")])}')
display(df_siniestros_agg.head(10))

Base agregada: df_siniestros_agg
Filas finales: 240,108
Asegurados unicos: 240,108
Columnas de tipo reclamacion agregadas: 8


,Afiliado_Id,n_reclamaciones,n_eventos,total_pagado,fecha_primera_reclamacion,fecha_ultima_reclamacion,rec_CIRUGIA,rec_CONSULTA MEDICA,rec_CONSULTAS,rec_EXAMENES MEDICOS,rec_HOSPITALIZACIONES COMPLEJAS,rec_HOSPITALIZACIONES SIMPLES,rec_LABORATORIO,rec_TRATAMIENTO DE CANCER
0,738294,8,10,3.131529e+06,10/04/2024,9/04/2024,0,4,0,4,0,0,0,0
1,738303,7,8,4.135152e+06,17/12/2024,9/03/2024,0,1,0,6,0,0,0,0
2,738305,7,8,8.020569e+05,13/04/2024,5/04/2024,0,6,0,1,0,0,0,0
3,738310,9,9,2.292458e+06,12/12/2024,6/12/2024,1,5,0,3,0,0,0,0
4,738318,3,3,3.929181e+05,15/07/2024,22/04/2024,0,0,0,3,0,0,0,0
5,738321,12,13,1.822294e+07,10/04/2024,9/04/2024,2,4,0,6,0,0,0,0
6,738348,9,9,3.267979e+06,1/08/2024,4/01/2024,0,5,0,4,0,0,0,0
7,738349,5,5,2.843624e+07,1/03/2024,6/04/2024,0,1,2,1,0,1,0,0
8,738358,12,13,1.821912e+06,13/08/2024,30/10/2024,0,11,0,1,0,0,0,0
9,738364,12,12,4.627797e+06,1/03/2024,7/03/2024,3,7,0,2,0,0,0,0


## 5. Integracion de las 3 bases
Unificamos exposición, sociodemográficos y siniestros en una sola base a nivel de afiliado y validamos calidad de la unión.

In [ ]:
# 1) Preparar exposición a nivel afiliado

df_exposicion_merge = df_exposicion_modelo.copy()
if 'Afiliado_Id' not in df_exposicion_merge.columns and 'Asegurado_Id' in df_exposicion_merge.columns:
    df_exposicion_merge = df_exposicion_merge.rename(columns={'Asegurado_Id': 'Afiliado_Id'})

df_exposicion_agg = df_exposicion_merge.groupby('Afiliado_Id', as_index=False).agg(
    dias_expuesto_total=('dias_expuesto', 'sum'),
    meses_expuesto_total=('meses_expuesto', 'sum'),
    n_polizas=('Poliza_Asegurado_Id', 'nunique')
)

# 2) Asegurar unicidad en sociodemográficos

df_socio_agg = df_sociodemograficos_limpio.drop_duplicates(subset=['Afiliado_Id']).copy()

# 3) Unión final (base maestra: sociodemográficos)

df_modelo_base = (
    df_socio_agg
    .merge(df_exposicion_agg, on='Afiliado_Id', how='left')
    .merge(df_siniestros_agg, on='Afiliado_Id', how='left')
)

# 4) Validaciones de calidad de la unión

print('=== Base final unificada: df_modelo_base ===')
print(f'Filas: {len(df_modelo_base):,}')
print(f'Afiliados unicos: {df_modelo_base["Afiliado_Id"].nunique():,}')
print(f'Duplicados por Afiliado_Id: {df_modelo_base.duplicated(subset=["Afiliado_Id"]).sum():,}')

resumen_nulos_final = pd.DataFrame({
    'nulos': df_modelo_base.isna().sum(),
    'pct_%': (df_modelo_base.isna().mean() * 100).round(2)
}).sort_values('pct_%', ascending=False)

print('\nTop 15 columnas con mayor porcentaje de nulos:')
display(resumen_nulos_final.head(15))

print('\nVista rápida de la base final:')
display(df_modelo_base.head(10))